In [1]:
import gym
#from gym.wrappers import Monitor
import random
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import matplotlib.pyplot as plt
import base64, io

import numpy as np
from collections import deque, namedtuple

# For visualization
from gym.wrappers.monitoring import video_recorder
from IPython.display import HTML
from IPython import display 
import glob

### 환경만들기 

`-` 환경을 만드는 방법은 아래와 같다. 

In [2]:
env = gym.make('LunarLander-v2',new_step_api=True)

- `new_step_api=True`를 쓰지 않으면 귀찮은 warning이 생기니까 시키는대로 해보자.. (무슨 역할인지는 정확하게 모르겠음)

`-` 환경에 대한 기본 정보를 조사하여 보자. 

In [3]:
env.observation_space

Box([-1.5       -1.5       -5.        -5.        -3.1415927 -5.
 -0.        -0.       ], [1.5       1.5       5.        5.        3.1415927 5.        1.
 1.       ], (8,), float32)

- 환경에 대한 상태는 모두 8개의 변수로 표현이 가능하다. 
- 각 변수의 범위는 (-1.5, 1.5), (-1.5, 1.5), (-5, 5), (-5, 5), (-3.14, 3.14), (-5, 5), (0, 1), (0, 1) 이다. 

In [4]:
env.action_space

Discrete(4)

- 이 환경에서 유효한 action은 4개이다. (=에이전트는 4개의 action을 할 수 있다.)

### 환경관찰 + 기록

`-` 환경관찰 

In [5]:
env.reset()

array([ 0.00527077,  1.401312  ,  0.533851  , -0.4270424 , -0.00610066,
       -0.12092521,  0.        ,  0.        ], dtype=float32)

In [6]:
env.step(0) # state, reward, done, _ 

(array([ 0.01054163,  1.3911275 ,  0.53313   , -0.4526785 , -0.01207623,
        -0.11952209,  0.        ,  0.        ], dtype=float32),
 -1.1571501945836644,
 False,
 False,
 {})

- 보상이 왜 정의되고 있는것이지? 

`-` memory 

(예비학습)

In [7]:
_memory=deque(maxlen=5)
_memory

deque([])

In [8]:
_memory

deque([])

In [9]:
state = env.reset()

In [10]:
action = env.action_space.sample() # agent.act(state,eps) 

In [87]:
next_state, reward, done, _ = env.step(action)

ValueError: too many values to unpack (expected 4)

In [12]:
# agent.step(state,action,reward,next_state,done) 

In [13]:
state = next_state 

In [ ]:
net

In [19]:
state

array([-2.9623031e-03,  1.3945472e+00, -1.4343588e-01, -3.7676364e-01,
        1.3528549e-03, -8.3909286e-03,  0.0000000e+00,  0.0000000e+00],
      dtype=float32)

In [17]:
qnetwork_local = QNetwork(8,4,0) # 8개의 상태 -> 4개의 액션중에 난 뭘해야해?

In [22]:
qnetwork_local.eval()
with torch.no_grad():
    action_values= qnetwork_local(torch.tensor(state))
qnetwork_local.train()

QNetwork(
  (layer1): Linear(in_features=8, out_features=128, bias=True)
  (layer2): Linear(in_features=128, out_features=64, bias=True)
  (layer3): Linear(in_features=64, out_features=32, bias=True)
  (layer4): Linear(in_features=32, out_features=4, bias=True)
)

In [27]:
action_values

tensor([ 0.1111, -0.0396,  0.1566, -0.1227])

In [25]:
np.argmax(action_values.numpy()) # 이상태에서는 action 2를 선택하는게 좋다.. 

2

In [62]:
next_states = torch.tensor([[ 4.7951e-04,  1.4172e+00,  4.8555e-02,  2.8079e-01, -5.4885e-04,
          -1.0999e-02,  0.0000e+00,  0.0000e+00],
         [ 4.7951e-04,  1.4172e+00,  4.8555e-02,  2.8079e-01, -5.4885e-04,
          -1.0999e-02,  0.0000e+00,  0.0000e+00]])

In [63]:
q_targets_next = qnetwork_target(next_states).detach().max(1)[0].reshape(-1,1)
q_targets_next

tensor([[0.1669],
        [0.1669]])

- 참고~ (뜻: action=0 이 좋아요, 왜? action=0의 q-value가 0.1669로 가장 높으니까요..)
```
qnetwork_target(next_states)
tensor([[ 0.1669,  0.0127,  0.1299, -0.1699],
        [ 0.1669,  0.0127,  0.1299, -0.1699]], grad_fn=<AddmmBackward0>)
```

In [81]:
qnetwork_target = QNetwork(8,4,1) # 8개의 상태 -> 4개의 액션중에 난 뭘해야해?
qnetwork_target

QNetwork(
  (layer1): Linear(in_features=8, out_features=128, bias=True)
  (layer2): Linear(in_features=128, out_features=64, bias=True)
  (layer3): Linear(in_features=64, out_features=32, bias=True)
  (layer4): Linear(in_features=32, out_features=4, bias=True)
)

In [82]:
states = next_states

In [83]:
actions = torch.tensor([[2],[2]])
actions

tensor([[2],
        [2]])

In [84]:
qnetwork_local(states).gather(1,actions)

tensor([[0.1400],
        [0.1400]], grad_fn=<GatherBackward0>)

In [85]:
qnetwork_local(states).gather(1, actions)

tensor([[0.1400],
        [0.1400]], grad_fn=<GatherBackward0>)

In [71]:
qnetwork_local(states).detach().max(1)[0].reshape(-1,1)

tensor([[0.1400],
        [0.1400]])

In [15]:
class QNetwork(nn.Module):
    """Actor (Policy) Model."""

    def __init__(self, state_size, action_size, seed):
        """Initialize parameters and build model.
        Params
        ======
            state_size (int): Dimension of each state
            action_size (int): Dimension of each action
            seed (int): Random seed
        """
        super(QNetwork, self).__init__()
        self.seed = torch.manual_seed(seed)
        ############################### TODO: YOUR CODE BELOW ##############################
        ### Q Network에 사용될 layers을 정의해주세요                                          ###
        ####################################################################################
        self.layer1 = nn.Linear(state_size, 128)
        self.layer2 = nn.Linear(128, 64)
        self.layer3 = nn.Linear(64, 32)
        self.layer4 = nn.Linear(32, action_size)
        ################################# END OF YOUR CODE #################################
        
    def forward(self, state):
        """Build a network that maps state -> action values."""
        
        ############################### TODO: YOUR CODE BELOW ##############################
        ###  Forward Pass를 구현해주세요                                                    ###
        ####################################################################################        
        x = F.relu(self.layer1(state))        
        x = F.relu(self.layer2(x))
        x = F.relu(self.layer3(x))
        x = self.layer4(x)
        ################################# END OF YOUR CODE #################################
        
        return x